In [1]:
import onnxruntime as ort
from jetbot import Robot, Camera
import numpy as np
import ipywidgets.widgets as widgets
from jetbot import bgr8_to_jpeg
import traitlets
from time import sleep
import cv2
import torch
# Camera should be positioned so that its bottom edge 
# cuts across the [2, 5, 8, /] buttons of the numpad, when the 
# robot is placed on the keyboard side, facing towards it.
class Model():
    def __init__(self, model_file, dbg=False):
        self.model = ort.InferenceSession(model_file, providers=['CPUExecutionProvider'])
        self.camera = Camera()
        self.dbg = dbg
        if self.dbg:
            self.widget = widgets.Image(format='jpeg', width=500, height=500)
            display(self.widget)
        else:
            self.widget = None

        
        #self.robot = Robot()
    
    def run(self, widget=False):
        # return np.array([[[1, 0]]]) # Mock backup :)
        image = np.frombuffer(self.camera.value, dtype=np.uint8)
        image = image.reshape((self.camera.height, self.camera.width, 3)).astype(np.float32)
        
        
        # PASTE BEGIN
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)#.reshape(3, 1, 1)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)#.reshape(3, 1, 1)
        
        
        #image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image[:, :, ::-1]

#         image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)

        image = image.astype(np.float32) / 255.0
        image = (image - mean) / std

        image = np.transpose(image, (2, 0, 1))

        image = np.array([image], dtype=np.float32)
#         image = torch.from_numpy(image)
#         # PASTE END
        
        input_name = self.model.get_inputs()[0].name
        
        res = self.model.run(None, {input_name: image})
        
        if widget:
            camera_link = traitlets.dlink((self.camera, 'value'), (self.widget, 'value'), transform=bgr8_to_jpeg)
            print('Output:', res)
        
        return res

In [2]:
class Driver():
    def __init__(self, model_path, refresh_delay=.1, speed=1, dbg=False):
        self.robot = Robot()
        self.model = Model(model_path, dbg=dbg)
        self.refresh_delay = refresh_delay
        self.speed = speed
        self.dbg = dbg
        self.steer = .15
        
        self.set_movement(0, 0)
        sleep(.1)
        self.set_movement(0, 0, stop=True)
        sleep(.1)
        self.set_movement(0, 0)
        sleep(.01)
        self.set_movement(0, 0, stop=True)
            
    
    def set_movement(self, x, y, stop=False):
        if stop:
            self.robot.left_motor.value  = 0
            self.robot.right_motor.value = 0
            return
        MAX_WHEEL_SPEED_FWD = .4
        MAX_WHEEL_SPEED_TURN = .25
        TURN_THRESHOLD = .19
        # Negative is fwd, left
        y = 1
        # x = 0
        if self.dbg:
            print('Move', x, y, end=', ')
        # x = abs(x)
        y *= -1
        x *= -1
        y = max(0, y)
       
        
        l = -.5 * y + self.steer * x
        r = -.5 * y - self.steer * x
        if True:
            if x < 0: 
                l = -.5 * y + self.steer * x
                r = -.5 * y - self.steer * x
            else:
                l = -.5 * y + self.steer * x * 1.1
                r = -.5 * y - self.steer * x * 1.1
        
        max_wheel_input = max(l, r)
        clamp_val = MAX_WHEEL_SPEED_FWD if abs(l - r) < TURN_THRESHOLD else MAX_WHEEL_SPEED_TURN
        if max_wheel_input > clamp_val:
            scaler = clamp_val / max_wheel_input
            l *= scaler
            r *= scaler
        
        l += .5 - .01
        r += .5
        
        l *= self.speed
        r *= self.speed
        self.robot.left_motor.value  = l
        self.robot.right_motor.value = r
        
        
    def run_loop(self):
        keep_going = True
        while keep_going:
            try:
                res = self.model.run(widget=self.dbg)
                # print(res[0][0])
                x, y = res[0][0]
                self.set_movement(x, y)
                sleep(self.refresh_delay)
            except KeyboardInterrupt:
                keep_going = False
                self.set_movement(0, 0, stop=True)
                
    def stop(self):
        self.set_movement(0, 0, stop=True)
        
    def set_speed(self, spd):
        self.speed = spd
        
    def set_steer(self, steer):
        self.steer = steer

In [3]:
d = Driver('./model_final_final_02_final_please_2_fr_final_maybe.onnx', refresh_delay=0, speed=0.4, dbg=False)

In [ ]:
d.run_loop()

In [5]:
d.stop()

In [10]:
d.set_speed(.4)

In [5]:
d.set_steer(.15)